In [ ]:
import os
os.environ['TPU_SKIP_MDS_QUERY'] = '1'

! pip install transformers datasets sacrebleu kagglehub --quiet

import os, random
import torch
from datasets import Dataset
from transformers import MT5Tokenizer, MT5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from sacrebleu import corpus_bleu, corpus_chrf
import kagglehub
from dataclasses import dataclass
from typing import Any, Dict, List

dataset_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset")
with open(os.path.join(dataset_path, "train.en"), encoding="utf-8") as f: english = [l.strip() for l in f][:2000]
with open(os. path.join(dataset_path, "train.ml"), encoding="utf-8") as f: malayalam = [l.strip() for l in f][:2000]
pairs = [(en, ml) for en, ml in zip(english, malayalam) if en and ml]
english, malayalam = zip(*pairs)

random.seed(42)
indices = list(range(len(english)))
random.shuffle(indices)
train_en = [english[i] for i in indices[:1200]]
train_ml = [malayalam[i] for i in indices[:1200]]
eval_en = [english[i] for i in indices[1200:1400]]
eval_ml = [malayalam[i] for i in indices[1200:1400]]
test_en = [english[i] for i in indices[1400:1600]]
test_ml = [malayalam[i] for i in indices[1400:1600]]

tokenizer = MT5Tokenizer.from_pretrained("google/mt5-base")

def preprocess(examples):
    inputs = [f"translate English to Malayalam: {src}" for src in examples["en"]]
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=128)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(list(examples["ml"]), padding="max_length", truncation=True, max_length=128)

    labels_ids = []
    for label_seq in labels["input_ids"]:
        labels_ids.append([lbl if lbl != tokenizer.pad_token_id else -100 for lbl in label_seq])

    model_inputs["labels"] = labels_ids
    return model_inputs

train_ds = Dataset.from_dict({"en": train_en, "ml": train_ml}). map(preprocess, batched=True, remove_columns=["en", "ml"])
eval_ds = Dataset.from_dict({"en": eval_en, "ml": eval_ml}).map(preprocess, batched=True, remove_columns=["en", "ml"])

model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")

@dataclass
class CustomSeq2SeqCollator:
    tokenizer: Any

    def __call__(self, batch: List[Dict[str, List[int]]]) -> Dict[str, Any]:
        input_ids = torch.tensor([item["input_ids"] for item in batch])
        attention_mask = torch.tensor([item["attention_mask"] for item in batch])
        labels = torch. tensor([item["labels"] for item in batch])
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

args = Seq2SeqTrainingArguments(
    output_dir="mt5-enml-results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    predict_with_generate=False,  # Disable for memory
    logging_steps=100,
    save_total_limit=1,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
    learning_rate=1e-4,
    remove_unused_columns=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=CustomSeq2SeqCollator(tokenizer=tokenizer)
)

print("Starting training on CPU/GPU (not TPU)...")
trainer.train()
print("Training complete!")

model.eval()
print("Evaluating on test set...")
inputs = tokenizer([f"translate English to Malayalam: {t}" for t in test_en], return_tensors="pt", padding=True, truncation=True, max_length=128)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
for k in inputs: inputs[k] = inputs[k]. to(device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)

bleu = corpus_bleu(predictions, [list(test_ml)])
chrf = corpus_chrf(predictions, [list(test_ml)])

report = [
    "mT5-base fine-tuned English→Malayalam Translation Report",
    "=" * 55,
    f"Test set size: {len(test_en)}",
    "",
    "Evaluation Metrics:",
    f"BLEU Score: {bleu. score:.2f}",
    f"chrF Score: {chrf.score:.2f}",
    f"Brevity Penalty: {bleu.bp:.3f}",
    "",
    "Sample Translations (first 5):",
    "-" * 55
]
for i in range(5):
    report += [f"{i+1}. English: {test_en[i]}", f"   MT5 Prediction: {predictions[i]}", f"   Reference: {test_ml[i]}", ""]

with open("mt5_enml_report.txt", "w", encoding="utf-8") as f: f.write('\n'.join(report))
print(f"\nReport saved to mt5_enml_report.txt")
print(f"BLEU: {bleu.score:.2f}, chrF: {chrf.score:.2f}")

from google.colab import files; files.download("mt5_enml_report.txt")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 35.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


100%|██████████| 407M/407M [00:02<00:00, 147MB/s]

Extracting files...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,12.219500
200,5.482200
300,4.710000
400,4.610800
500,4.379100
600,4.257100
700,4.105900
800,4.153300
900,4.031200
1000,4.102700


Training complete!
Evaluating on test set...

Report saved to mt5_enml_report.txt
BLEU: 1.03, chrF: 19.89


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
! pip install transformers sacrebleu kagglehub --quiet

import os, random
import torch
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
from sacrebleu import corpus_bleu, corpus_chrf
import kagglehub

# Load dataset
dataset_path = kagglehub.dataset_download("parvmodi/english-to-malayalam-machine-translation-dataset")
with open(os.path.join(dataset_path, "train.en"), encoding="utf-8") as f: english = [l.strip() for l in f][:2000]
with open(os. path.join(dataset_path, "train.ml"), encoding="utf-8") as f: malayalam = [l.strip() for l in f][:2000]
pairs = [(en, ml) for en, ml in zip(english, malayalam) if en and ml]
english, malayalam = zip(*pairs)

random.seed(42)
indices = list(range(len(english)))
random.shuffle(indices)
test_en = [english[i] for i in indices[1400:1600]]
test_ml = [malayalam[i] for i in indices[1400:1600]]

# Load trained model
print("Loading trained mT5 model from ./mt5-enml-results...")
model = MT5ForConditionalGeneration.from_pretrained("./mt5-enml-results")
tokenizer = MT5Tokenizer.from_pretrained("./mt5-enml-results")
print("✅ Model loaded successfully!\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# Generate predictions
print(f"Generating predictions on {len(test_en)} test samples...")
predictions = []
batch_size = 4
for i in range(0, len(test_en), batch_size):
    batch = test_en[i:i+batch_size]
    inputs = tokenizer([f"translate English to Malayalam: {t}" for t in batch],
                       return_tensors="pt", padding=True, truncation=True, max_length=128). to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4)
    predictions += tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Calculate metrics
bleu = corpus_bleu(predictions, [list(test_ml)])
chrf = corpus_chrf(predictions, [list(test_ml)])

print(f"✅ Evaluation complete!\n")
print(f"BLEU: {bleu.score:.2f} | chrF: {chrf.score:.2f} | Brevity Penalty: {bleu.bp:.3f}\n")

# Generate detailed report
report = [
    "=" * 75,
    "mT5-base Fine-tuned English→Malayalam Translation Evaluation Report",
    "=" * 75,
    "",
    f"Model: google/mt5-base (fine-tuned)",
    f"Test Set Size: {len(test_en)} sentence pairs",
    "",
    "EVALUATION METRICS:",
    "-" * 75,
    f"Corpus BLEU Score:     {bleu.score:.2f}",
    f"chrF Score:            {chrf.score:.2f}",
    f"Brevity Penalty:       {bleu.bp:.3f}",
    f"1-gram precision:      {bleu.precisions[0]:.2f}",
    f"2-gram precision:      {bleu.precisions[1]:.2f}",
    f"3-gram precision:      {bleu.precisions[2]:.2f}",
    f"4-gram precision:      {bleu.precisions[3]:.2f}",
    "",
    "SAMPLE TRANSLATIONS (First 10):",
    "-" * 75,
]

for i in range(min(10, len(test_en))):
    report += [
        f"\n{i+1}. English Source:",
        f"   {test_en[i]}",
        f"",
        f"   Model Prediction:",
        f"   {predictions[i]}",
        f"",
        f"   Reference Translation:",
        f"   {test_ml[i]}"
    ]

report += ["", "-" * 75, "END OF REPORT"]

# Save report
with open("mt5_evaluation_report.txt", "w", encoding="utf-8") as f:
    f.write('\n'.join(report))

print("✅ Report saved to: mt5_evaluation_report.txt\n")

# Display summary
print("=" * 75)
print("EVALUATION SUMMARY")
print("=" * 75)
print(f"Test Samples:          {len(test_en)}")
print(f"BLEU Score:            {bleu.score:.2f}")
print(f"chrF Score:            {chrf.score:.2f}")
print(f"Brevity Penalty:       {bleu.bp:.3f}")
print("=" * 75 + "\n")

# Download report
from google.colab import files
files.download("mt5_evaluation_report.txt")
print("✅ Download started!")

Using Colab cache for faster access to the 'english-to-malayalam-machine-translation-dataset' dataset.
Loading trained mT5 model from ./mt5-enml-results...


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory ./mt5-enml-results.